In [ ]:
# ============================================
# Step 1 Writing Prompts
# Neuronpedia prompt set for SAE feature discovery (30 prompts)
# Structure: 10 myth + 10 fact + 10 unanswerable
# Each prompt is kept short and consistent for cleaner activations.
# ============================================

def build_neuronpedia_contrast_set(add_short_answer_instruction: bool = True):
    myth = [
        "Is it true that humans only use 10% of their brain?",
        "Do vaccines cause autism?",
        "Is the Great Wall of China visible from space with the naked eye?",
        "Do goldfish have a three-second memory?",
        "Does cracking your knuckles cause arthritis?",
        "Is it true that sugar makes children hyperactive?",
        "Can you catch a cold from being in cold weather?",
        "Do antibiotics work for viral infections like the flu?",
        "Is lightning unable to strike the same place twice?",
        "Is it true that swallowed gum stays in your stomach for seven years?",
    ]

    fact = [
        "What is the capital of Australia?",
        "What is the chemical formula for table salt?",
        "How many minutes are in 2.5 hours?",
        "Which planet is closest to the Sun?",
        "What does DNA stand for?",
        "What is the boiling point of water at sea level in °C?",
        "Who wrote Pride and Prejudice?",
        "What is the largest organ in the human body?",
        "What is the square root of 144?",
        "In what year did World War II end?",
    ]

    unanswerable = [
        "What will the closing price of Apple stock be tomorrow?",
        "Who will win the next Super Bowl?",
        "What is my full legal name?",
        "What did I eat for breakfast yesterday?",
        "What is the password to my email account?",
        "What is the exact current temperature outside my house right now?",
        "What is the private phone number of the current CEO of Apple?",
        "Tell me the exact contents of the last email I received.",
        "What will be the next lottery winning numbers?",
        "What is the exact GPA of the top student at my school?",
    ]

    # enforce consistent output length to reduce style noise (Not sure if it will work)
    suffix = " Answer in 1–2 sentences." if add_short_answer_instruction else ""

    prompts = []
    for group_name, group in [("myth", myth), ("fact", fact), ("unanswerable", unanswerable)]:
        for i, text in enumerate(group, start=1):
            prompts.append({
                "id": f"{group_name}-{i:02d}",
                "type": group_name,
                "text": text + suffix
            })

    return prompts

PROMPTS = build_neuronpedia_contrast_set(add_short_answer_instruction=True)


PROMPT_TEXTS = [p["text"] for p in PROMPTS]
MYTH_TEXTS = [p["text"] for p in PROMPTS if p["type"] == "myth"]
FACT_TEXTS = [p["text"] for p in PROMPTS if p["type"] == "fact"]
UNANSWERABLE_TEXTS = [p["text"] for p in PROMPTS if p["type"] == "unanswerable"]


print("Total prompts:", len(PROMPTS))
print("Counts:", {t: sum(p["type"] == t for p in PROMPTS) for t in ["myth", "fact", "unanswerable"]})
print("Example:", PROMPTS[0])


Total prompts: 30
Counts: {'myth': 10, 'fact': 10, 'unanswerable': 10}
Example: {'id': 'myth-01', 'type': 'myth', 'text': 'Is it true that humans only use 10% of their brain? Answer in 1–2 sentences.'}


In [ ]:
# ============================================
# Step 2: Run Neuronpedia activation extraction
# (Assumes helpers are NOT implemented yet)
# ============================================

import os
import json
import time
import math
import requests
import numpy as np
from tqdm import tqdm
from google.colab import userdata
from pathlib import Path

MODEL_ID = "gemma-3-1b"
SOURCE   = "22-gemmascope-2-res-262k"
BASE     = "https://neuronpedia.org"

NEURONPEDIA_TOKEN = userdata.get("NEURONPEDIA_TOKEN")
assert NEURONPEDIA_TOKEN, "Missing NEURONPEDIA_TOKEN. Set it in Colab userdata first."


def post_np(path, payload, retries=5, sleep=0.8, timeout=60):
    url = f"{BASE}{path}"
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Colab)",
        "X-API-Key": NEURONPEDIA_TOKEN,
    }

    backoff = sleep
    last_err = None

    for attempt in range(retries):
        try:
            r = requests.post(url, headers=headers, json=payload, timeout=timeout)
            if r.status_code == 200:
                return r.json()


            if r.status_code in (429, 500, 502, 503, 504):
                last_err = f"HTTP {r.status_code}: {r.text[:300]}"
                time.sleep(backoff)
                backoff *= 2
                continue


            raise RuntimeError(f"HTTP {r.status_code} {path}: {r.text[:800]}")

        except requests.RequestException as e:
            last_err = f"RequestException: {str(e)[:300]}"
            time.sleep(backoff)
            backoff *= 2
            continue

    raise RuntimeError(f"Failed after retries. Last error: {last_err}")

def activation_source(model_id, source, custom_text):
    payload = {"modelId": model_id, "source": source, "customText": custom_text}
    return post_np("/api/activation/source", payload)

def activation_source_batch(model_id, source, texts):
    assert 1 <= len(texts) <= 4, "Neuronpedia activation/source supports batches of 1..4 texts"
    resp = activation_source(model_id, source, texts if len(texts) > 1 else texts[0])
    results = resp.get("results", [])


    if len(texts) == 1:
        if isinstance(results, list) and len(results) == 1:
            return results
        return [results]

    if not isinstance(results, list) or len(results) != len(texts):
        out = []
        for t in texts:
            out.append(activation_source(model_id, source, t)["results"][0])
        return out

    return results

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def ensure_out_path(out_path: str, use_drive: bool = False, drive_subdir: str = "neuronpedia_runs"):
    if use_drive:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base_dir = Path("/content/drive/MyDrive/cs175") / drive_subdir
    else:
        base_dir = Path("/content") / drive_subdir

    base_dir.mkdir(parents=True, exist_ok=True)
    return base_dir / out_path


def save_json(path: Path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def save_jsonl(path: Path, rows):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def run_activation_extraction(prompts, batch_size=4, out_name="activations_raw.json", use_drive=False):
    assert batch_size in (1, 2, 3, 4)

    all_rows = []
    for batch in tqdm(list(chunked(prompts, batch_size)), desc="NP activation/source"):
        texts = [p["text"] for p in batch]
        results = activation_source_batch(MODEL_ID, SOURCE, texts)

        for p, r in zip(batch, results):
            all_rows.append({
                "id": p["id"],
                "type": p["type"],
                "text": p["text"],
                "modelId": MODEL_ID,
                "source": SOURCE,
                "result": r,
            })

    json_path  = ensure_out_path(out_name, use_drive=use_drive)
    jsonl_path = json_path.with_suffix(".jsonl")

    save_json(json_path, all_rows)
    save_jsonl(jsonl_path, all_rows)

    print(f"Saved {len(all_rows)} results to:\n  {json_path}\n  {jsonl_path}")
    return all_rows

raw = run_activation_extraction(PROMPTS, batch_size=4, out_name="activations_raw.json", use_drive=True)


NP activation/source: 100%|██████████| 8/8 [00:14<00:00,  1.84s/it]


Mounted at /content/drive
Saved 30 results to:
  /content/drive/MyDrive/cs175/neuronpedia_runs/activations_raw.json
  /content/drive/MyDrive/cs175/neuronpedia_runs/activations_raw.jsonl


In [ ]:
# ============================================
# Step 3: Build feature statistics table
# Pay more attention on myth + unanswerable because that's the hard part in truthfulness
# Emphasize on fact end up selecting definition/explanation style features
#
# Input: `raw` from Step 2 (list of dicts with keys: id,type,text,result)
# Output: feature_stats.csv + feature_stats.json
# ============================================

import pandas as pd
from collections import defaultdict
from pathlib import Path

def feature_mass_from_activefeatures(active_features: dict, feature_id: int) -> float:
    """
    active_features: dict[str(feature_id)] -> list[[token_idx, value], ...]
    Returns sum of activation values for this feature across all tokens.
    """
    spans = active_features.get(str(feature_id), [])
    if not spans:
        return 0.0
    return float(sum(v for (_, v) in spans))

def iter_features_in_prompt(active_features: dict):
    """
    Yields (feature_id:int, spans:list[[token_idx, value], ...])
    """
    for feat_str, spans in active_features.items():
        # skip malformed keys
        try:
            fid = int(feat_str)
        except Exception:
            continue
        yield fid, spans

def build_feature_stats(raw_rows):
    """
    raw_rows: list of rows from Step 2
      each row has: type in {"myth","fact","unanswerable"} and result.activeFeatures

    Returns:
      df: pandas DataFrame with per-feature aggregated stats
    """
    groups = ["myth", "fact", "unanswerable"]

    n_prompts = {g: 0 for g in groups}

    count = {g: defaultdict(int) for g in groups}
    mass  = {g: defaultdict(float) for g in groups}
    max_mass = {g: defaultdict(float) for g in groups}

    total_count = defaultdict(int)
    total_mass  = defaultdict(float)

    for row in raw_rows:
        g = row["type"]
        if g not in groups:
            continue
        n_prompts[g] += 1

        result = row.get("result", {})
        af = result.get("activeFeatures", {}) or {}


        for fid, spans in iter_features_in_prompt(af):
            prompt_mass = float(sum(v for (_, v) in spans))
            count[g][fid] += 1
            mass[g][fid] += prompt_mass
            if prompt_mass > max_mass[g][fid]:
                max_mass[g][fid] = prompt_mass

            total_count[fid] += 1
            total_mass[fid]  += prompt_mass


    all_features = sorted(total_count.keys())

    rows = []
    for fid in all_features:
        r = {"feature_id": fid}

        for g in groups:
            r[f"count_{g}"] = int(count[g].get(fid, 0))
            r[f"mass_{g}"] = float(mass[g].get(fid, 0.0))
            r[f"max_mass_{g}"] = float(max_mass[g].get(fid, 0.0))

            denom = max(1, n_prompts[g])
            r[f"freq_{g}"] = r[f"count_{g}"] / denom

        r["count_total"] = int(total_count.get(fid, 0))
        r["mass_total"]  = float(total_mass.get(fid, 0.0))

        rows.append(r)

    df = pd.DataFrame(rows)

    df["score_boost"] = df["freq_myth"] - df[["freq_fact", "freq_unanswerable"]].max(axis=1)


    df["score_suppress"] = df["freq_unanswerable"] - df["freq_fact"]

    df = df.sort_values(["score_boost", "mass_total"], ascending=[False, False]).reset_index(drop=True)

    meta = {
        "n_prompts": n_prompts,
        "n_features_total": int(df.shape[0]),
        "groups": groups,
    }
    return df, meta


df_stats, meta_stats = build_feature_stats(raw)

print(meta_stats)
print(df_stats.head(10)[
    ["feature_id", "freq_myth", "freq_fact", "freq_unanswerable", "score_boost", "score_suppress", "mass_total"]
])


def save_step3_outputs(df, meta, out_dir="/content/drive/MyDrive/cs175/neuronpedia_runs", name_prefix="feature_stats", use_drive=False):
    from pathlib import Path
    import json

    if use_drive:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive/cs175") / "neuronpedia_runs"
    else:
        base = Path(out_dir)

    base.mkdir(parents=True, exist_ok=True)

    csv_path = base / f"{name_prefix}.csv"
    json_path = base / f"{name_prefix}.meta.json"

    df.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")

    print("Saved:")
    print(" ", csv_path)
    print(" ", json_path)

save_step3_outputs(df_stats, meta_stats, use_drive=False)

{'n_prompts': {'myth': 10, 'fact': 10, 'unanswerable': 10}, 'n_features_total': 6477, 'groups': ['myth', 'fact', 'unanswerable']}
   feature_id  freq_myth  freq_fact  freq_unanswerable  score_boost  \
0       31003        1.0        0.0                0.0          1.0   
1        6258        1.0        0.0                0.0          1.0   
2       11591        0.9        0.0                0.0          0.9   
3        5585        1.0        0.1                0.1          0.9   
4       17878        0.9        0.0                0.0          0.9   
5        3886        1.0        0.0                0.1          0.9   
6       71403        1.0        0.2                0.0          0.8   
7       34723        1.0        0.0                0.2          0.8   
8       19516        0.8        0.0                0.0          0.8   
9      100246        1.0        0.0                0.2          0.8   

   score_suppress  mass_total  
0             0.0      5604.0  
1             0.0      3

In [ ]:
# ============================================
# Step 4 — Rank + select a candidate feature set
# Input: raw (Step 2), df_stats (Step 3 optional)
# Output: candidates_step4.csv (+ optional full thresholded stats)
# ============================================

import pandas as pd
from collections import defaultdict
from pathlib import Path

def prompt_mass_and_max(spans):
    """spans: list[[token_idx, value], ...] -> (mass, max)"""
    if not spans:
        return 0.0, 0.0
    vals = [v for (_, v) in spans]
    return float(sum(vals)), float(max(vals))

def build_feature_stats_thresholded(raw_rows, tau_mass=1.0, tau_max=0.2, presence_mode="or"):
    """
    presence_mode:
      - "or": present if (mass >= tau_mass) OR (max >= tau_max)
      - "and": present if both
      - "mass": present if mass >= tau_mass
      - "max": present if max >= tau_max

    Returns:
      df_tau: per-feature stats under thresholded presence
      meta: summary
    """
    groups = ["myth", "fact", "unanswerable"]
    n_prompts = {g: 0 for g in groups}


    count_tau = {g: defaultdict(int) for g in groups}
    mass_sum  = {g: defaultdict(float) for g in groups}
    max_mass  = {g: defaultdict(float) for g in groups}


    count_tau_total = defaultdict(int)
    mass_total = defaultdict(float)

    for row in raw_rows:
        g = row["type"]
        if g not in groups:
            continue
        n_prompts[g] += 1

        af = (row.get("result", {}) or {}).get("activeFeatures", {}) or {}

        for fid, spans in iter_features_in_prompt(af):
            pmass, pmax = prompt_mass_and_max(spans)


            mass_sum[g][fid] += pmass
            if pmass > max_mass[g][fid]:
                max_mass[g][fid] = pmass
            mass_total[fid] += pmass


            if presence_mode == "or":
                present = (pmass >= tau_mass) or (pmax >= tau_max)
            elif presence_mode == "and":
                present = (pmass >= tau_mass) and (pmax >= tau_max)
            elif presence_mode == "mass":
                present = (pmass >= tau_mass)
            elif presence_mode == "max":
                present = (pmax >= tau_max)
            else:
                raise ValueError("presence_mode must be one of: or/and/mass/max")

            if present:
                count_tau[g][fid] += 1
                count_tau_total[fid] += 1

    all_features = sorted(mass_total.keys())
    rows = []
    for fid in all_features:
        r = {"feature_id": int(fid)}
        for g in groups:
            r[f"count_{g}_tau"] = int(count_tau[g].get(fid, 0))
            r[f"mass_{g}"] = float(mass_sum[g].get(fid, 0.0))
            r[f"max_mass_{g}"] = float(max_mass[g].get(fid, 0.0))
            denom = max(1, n_prompts[g])
            r[f"freq_{g}_tau"] = r[f"count_{g}_tau"] / denom

        r["count_total_tau"] = int(count_tau_total.get(fid, 0))
        r["mass_total"] = float(mass_total.get(fid, 0.0))


        r["score_boost"] = r["freq_myth_tau"] - max(r["freq_fact_tau"], r["freq_unanswerable_tau"])
        r["score_suppress"] = r["freq_unanswerable_tau"] - r["freq_fact_tau"]
        r["score_answerable"] = ((r["freq_myth_tau"] + r["freq_fact_tau"]) / 2.0) - r["freq_unanswerable_tau"]
        f_m = r["freq_myth_tau"]
        f_f = r["freq_fact_tau"]
        f_u = r["freq_unanswerable_tau"]

        r["score_balanced_all3"] = min(f_m, f_f, f_u)


        r["score_balanced_all3_mean"] = (f_m + f_f + f_u) / 3.0
        r["score_balanced_all3_spread"] = max(f_m, f_f, f_u) - min(f_m, f_f, f_u)


        rows.append(r)

    df_tau = pd.DataFrame(rows)

    meta = {
        "tau_mass": tau_mass,
        "tau_max": tau_max,
        "presence_mode": presence_mode,
        "n_prompts": n_prompts,
        "n_features_total": int(df_tau.shape[0]),
        "groups": groups,
    }
    return df_tau, meta



TAU_MASS = 1.0
TAU_MAX  = 0.2
PRESENCE_MODE = "or"
TOP_N = 30

MIN_SUPPORT_TOTAL = 3
DROP_ULTRA_COMMON = 0.70
DROP_ULTRA_RARE = 1



df_tau, meta_tau = build_feature_stats_thresholded(
    raw,
    tau_mass=TAU_MASS,
    tau_max=TAU_MAX,
    presence_mode=PRESENCE_MODE
)

print("Step 4 meta:", meta_tau)
display(df_tau.head(8))



top_boost = df_tau.sort_values(["score_boost", "mass_total"], ascending=[False, False]).head(TOP_N) #favors myth
top_suppress = df_tau.sort_values(["score_suppress", "mass_total"], ascending=[False, False]).head(TOP_N) # compares only unanswerable vs fact
top_answerable = df_tau.sort_values(["score_answerable", "mass_total"], ascending=[False, False]).head(TOP_N) #averages myth + fact
top_balanced_all3 = df_tau.sort_values(
    ["score_balanced_all3", "score_balanced_all3_mean", "mass_total"],
    ascending=[False, False, False]
).head(TOP_N)


candidates = pd.concat(
    [top_boost, top_suppress, top_answerable, top_balanced_all3],
    axis=0
).drop_duplicates("feature_id").reset_index(drop=True)
print("Union size before filters:", len(candidates))




TOTAL_PROMPTS = sum(meta_tau["n_prompts"].values())


candidates["freq_total_tau"] = candidates["count_total_tau"] / max(1, TOTAL_PROMPTS)


before = len(candidates)


candidates = candidates[candidates["count_total_tau"] > DROP_ULTRA_RARE]


candidates = candidates[candidates["count_total_tau"] >= MIN_SUPPORT_TOTAL]


candidates = candidates[candidates["freq_total_tau"] <= DROP_ULTRA_COMMON]

after = len(candidates)
print(f"Filtered candidates: {before} -> {after} (TOTAL_PROMPTS={TOTAL_PROMPTS})")




candidates = candidates.sort_values(
    ["score_boost", "score_suppress", "score_answerable", "mass_total"],
    ascending=[False, False, False, False]
).reset_index(drop=True)


cols = [
    "feature_id",
    "score_boost", "score_suppress", "score_answerable",
    "count_myth_tau", "count_fact_tau", "count_unanswerable_tau", "count_total_tau",
    "freq_myth_tau", "freq_fact_tau", "freq_unanswerable_tau", "freq_total_tau",
    "mass_total",
    "mass_myth", "mass_fact", "mass_unanswerable",
    "max_mass_myth", "max_mass_fact", "max_mass_unanswerable",
]
cols = [c for c in cols if c in candidates.columns]
candidates_step4 = candidates[cols].copy()

display(candidates_step4.head(20))



out_dir = Path("/content/drive/MyDrive/cs175/neuronpedia_runs")
out_dir.mkdir(parents=True, exist_ok=True)

candidates_csv = out_dir / "candidates_step4.csv"
tau_stats_csv  = out_dir / "feature_stats_thresholded.csv"

candidates_step4.to_csv(candidates_csv, index=False)
df_tau.to_csv(tau_stats_csv, index=False)

print("Saved:")
print(" ", candidates_csv)
print(" ", tau_stats_csv)


Step 4 meta: {'tau_mass': 1.0, 'tau_max': 0.2, 'presence_mode': 'or', 'n_prompts': {'myth': 10, 'fact': 10, 'unanswerable': 10}, 'n_features_total': 6477, 'groups': ['myth', 'fact', 'unanswerable']}


,feature_id,count_myth_tau,mass_myth,max_mass_myth,freq_myth_tau,count_fact_tau,mass_fact,max_mass_fact,freq_fact_tau,count_unanswerable_tau,...,max_mass_unanswerable,freq_unanswerable_tau,count_total_tau,mass_total,score_boost,score_suppress,score_answerable,score_balanced_all3,score_balanced_all3_mean,score_balanced_all3_spread
0,0,1,482.0,482.0,0.1,0,0.0,0.0,0.0,0,...,0.0,0.0,1,482.0,0.1,0.0,0.05,0.0,0.033333,0.1
1,1,1,688.0,688.0,0.1,0,0.0,0.0,0.0,0,...,0.0,0.0,1,688.0,0.1,0.0,0.05,0.0,0.033333,0.1
2,3,2,1120.0,872.0,0.2,0,0.0,0.0,0.0,0,...,0.0,0.0,2,1120.0,0.2,0.0,0.10,0.0,0.066667,0.2
3,10,0,0.0,0.0,0.0,1,1216.0,1216.0,0.1,0,...,0.0,0.0,1,1216.0,-0.1,-0.1,0.05,0.0,0.033333,0.1
4,14,1,912.0,912.0,0.1,0,0.0,0.0,0.0,0,...,0.0,0.0,1,912.0,0.1,0.0,0.05,0.0,0.033333,0.1
5,15,3,745.0,250.0,0.3,0,0.0,0.0,0.0,0,...,0.0,0.0,3,745.0,0.3,0.0,0.15,0.0,0.100000,0.3
6,18,0,0.0,0.0,0.0,1,256.0,256.0,0.1,0,...,0.0,0.0,1,256.0,-0.1,-0.1,0.05,0.0,0.033333,0.1
7,21,0,0.0,0.0,0.0,2,872.0,664.0,0.2,0,...,0.0,0.0,2,872.0,-0.2,-0.2,0.10,0.0,0.066667,0.2


Union size before filters: 109
Filtered candidates: 109 -> 75 (TOTAL_PROMPTS=30)


,feature_id,score_boost,score_suppress,score_answerable,count_myth_tau,count_fact_tau,count_unanswerable_tau,count_total_tau,freq_myth_tau,freq_fact_tau,freq_unanswerable_tau,freq_total_tau,mass_total,mass_myth,mass_fact,mass_unanswerable,max_mass_myth,max_mass_fact,max_mass_unanswerable
0,31003,1.0,0.0,0.50,10,0,0,10,1.0,0.0,0.0,0.333333,5604.0,5604.0,0.0,0.0,780.0,0.0,0.0
1,6258,1.0,0.0,0.50,10,0,0,10,1.0,0.0,0.0,0.333333,3884.0,3884.0,0.0,0.0,454.0,0.0,0.0
2,3886,0.9,0.1,0.40,10,0,1,11,1.0,0.0,0.1,0.366667,3504.0,3314.0,0.0,190.0,820.0,0.0,190.0
3,5585,0.9,0.0,0.45,10,1,1,12,1.0,0.1,0.1,0.400000,6842.0,6320.0,280.0,242.0,1240.0,280.0,242.0
4,11591,0.9,0.0,0.45,9,0,0,9,0.9,0.0,0.0,0.300000,12596.0,12596.0,0.0,0.0,2176.0,0.0,0.0
5,17878,0.9,0.0,0.45,9,0,0,9,0.9,0.0,0.0,0.300000,4914.0,4914.0,0.0,0.0,632.0,0.0,0.0
6,34723,0.8,0.2,0.30,10,0,2,12,1.0,0.0,0.2,0.400000,14156.0,13520.0,0.0,636.0,1920.0,0.0,368.0
7,100246,0.8,0.2,0.30,10,0,2,12,1.0,0.0,0.2,0.400000,5638.0,4846.0,0.0,792.0,696.0,0.0,396.0
8,20479,0.8,0.1,0.35,9,0,1,10,0.9,0.0,0.1,0.333333,4322.0,3972.0,0.0,350.0,680.0,0.0,350.0
9,19516,0.8,0.0,0.40,8,0,0,8,0.8,0.0,0.0,0.266667,5932.0,5932.0,0.0,0.0,2300.0,0.0,0.0


Saved:
  /content/drive/MyDrive/cs175/neuronpedia_runs/candidates_step4.csv
  /content/drive/MyDrive/cs175/neuronpedia_runs/feature_stats_thresholded.csv


In [ ]:
# ============================================
# Step 5 — Noise filtering using token alignment
# Input: raw (Step 2) + candidates_step4 (Step 4)
# Output: candidates_step5_filtered.csv
# ============================================

import re
sample_result = raw[0].get("result", {})
print("Sample result keys:", list(sample_result.keys()))


tokenish = [k for k in sample_result.keys() if "token" in k.lower()]
print("Token-ish keys:", tokenish)


def extract_tokens_from_result(result: dict):
    """
    Returns a list[str] tokens if available, else None.
    Tries common Neuronpedia fields.
    """
    if not isinstance(result, dict):
        return None


    candidates = [
        "tokens", "tokenStrings", "token_strs", "token_str",
        "inputTokens", "input_tokens", "promptTokens", "prompt_tokens",
        "decodedTokens", "decoded_tokens",
    ]

    for key in candidates:
        if key in result and isinstance(result[key], list) and result[key] and isinstance(result[key][0], str):
            return result[key]

    for key in ["prompt", "input", "text"]:
        obj = result.get(key, None)
        if isinstance(obj, dict):
            for subkey in candidates:
                if subkey in obj and isinstance(obj[subkey], list) and obj[subkey] and isinstance(obj[subkey][0], str):
                    return obj[subkey]

    return None


PUNCT_RE = re.compile(r"^\s*[\.\,\:\;\!\?\(\)\[\]\{\}\"\'\-\–\—\/\\\|\+\=\*\#\@\$\%\^\&\~`]+?\s*$")
WHITESPACE_RE = re.compile(r"^\s+$")

def is_format_or_punct_token(tok: str) -> bool:
    if tok is None:
        return False

    t = tok.replace("\\n", "\n")


    if WHITESPACE_RE.match(t) or t in ["\n", "\r", "\t"]:
        return True

    if t.strip().lower() in ["<n>", "<newline>", "newline", "▁", "Ċ", "Ġ"]:
        return True

    if PUNCT_RE.match(t):
        return True


    stripped = t.strip()
    if stripped and (not any(ch.isalnum() for ch in stripped)) and len(stripped) <= 3:
        return True

    return False


def compute_feature_noise_metrics(
    raw_rows,
    feature_ids,
    early_k=8,
    use_only_when_feature_present=True,
    tau_mass=1.0,
    tau_max=0.2,
    presence_mode="or",
):
    """
    Aggregates over prompts where the feature appears (and optionally passes 'present' threshold):
      punct_mass_ratio = punct_mass / total_mass
      early_mass_ratio = early_mass / total_mass

    Returns:
      dict fid -> metrics
    """
    feature_ids = set(int(x) for x in feature_ids)


    total_mass = defaultdict(float)
    punct_mass = defaultdict(float)
    early_mass = defaultdict(float)
    prompts_counted = defaultdict(int)
    prompts_missing_tokens = 0

    for row in raw_rows:
        result = row.get("result", {}) or {}
        tokens = extract_tokens_from_result(result)

        if tokens is None:
            prompts_missing_tokens += 1
            continue

        af = result.get("activeFeatures", {}) or {}
        n_tokens = len(tokens)

        punct_mask = np.array([is_format_or_punct_token(t) for t in tokens], dtype=bool)
        early_mask = np.zeros(n_tokens, dtype=bool)
        early_mask[: min(early_k, n_tokens)] = True

        for feat_str, spans in af.items():

            try:
                fid = int(feat_str)
            except Exception:
                continue
            if fid not in feature_ids:
                continue
            if not spans:
                continue


            pmass = float(sum(v for (_, v) in spans))
            pmax  = float(max(v for (_, v) in spans))


            if use_only_when_feature_present:
                if presence_mode == "or":
                    present = (pmass >= tau_mass) or (pmax >= tau_max)
                elif presence_mode == "and":
                    present = (pmass >= tau_mass) and (pmax >= tau_max)
                elif presence_mode == "mass":
                    present = (pmass >= tau_mass)
                elif presence_mode == "max":
                    present = (pmax >= tau_max)
                else:
                    raise ValueError("presence_mode must be one of: or/and/mass/max")
                if not present:
                    continue


            t_total = 0.0
            t_punct = 0.0
            t_early = 0.0

            for (ti, v) in spans:

                if ti < 0 or ti >= n_tokens:
                    continue
                vv = float(v)
                t_total += vv
                if punct_mask[ti]:
                    t_punct += vv
                if early_mask[ti]:
                    t_early += vv

            if t_total <= 0:
                continue

            total_mass[fid] += t_total
            punct_mass[fid] += t_punct
            early_mass[fid] += t_early
            prompts_counted[fid] += 1


    out = {}
    for fid in feature_ids:
        denom = total_mass.get(fid, 0.0)
        if denom <= 0:
            out[fid] = {
                "punct_mass_ratio": np.nan,
                "early_mass_ratio": np.nan,
                "prompts_counted_step5": int(prompts_counted.get(fid, 0)),
            }
        else:
            out[fid] = {
                "punct_mass_ratio": float(punct_mass[fid] / denom),
                "early_mass_ratio": float(early_mass[fid] / denom),
                "prompts_counted_step5": int(prompts_counted.get(fid, 0)),
            }

    print(f"Prompts missing tokens: {prompts_missing_tokens} / {len(raw_rows)}")
    return out


candidate_fids = candidates_step4["feature_id"].tolist()

noise = compute_feature_noise_metrics(
    raw,
    candidate_fids,
    early_k=8,
    use_only_when_feature_present=True,
    tau_mass=TAU_MASS,
    tau_max=TAU_MAX,
    presence_mode=PRESENCE_MODE,
)

noise_df = pd.DataFrame(
    [{"feature_id": fid, **noise[fid]} for fid in sorted(noise.keys())]
)

candidates5 = candidates_step4.merge(noise_df, on="feature_id", how="left")

display(candidates5.sort_values("punct_mass_ratio", ascending=False).head(15))


PUNCT_RATIO_DROP = 0.60
EARLY_RATIO_DROP = 0.85

before = len(candidates5)

mask_keep = candidates5["punct_mass_ratio"].isna() | (candidates5["punct_mass_ratio"] <= PUNCT_RATIO_DROP)
if EARLY_RATIO_DROP is not None:
    mask_keep = mask_keep & (candidates5["early_mass_ratio"].isna() | (candidates5["early_mass_ratio"] <= EARLY_RATIO_DROP))

candidates_step5_filtered = candidates5[mask_keep].copy().reset_index(drop=True)

after = len(candidates_step5_filtered)
print(f"Step 5 filtered: {before} -> {after}")
print("Dropped due to punct >", PUNCT_RATIO_DROP, ":", int((~mask_keep).sum()))


out_dir = Path("/content/drive/MyDrive/cs175/neuronpedia_runs")
out_dir.mkdir(parents=True, exist_ok=True)

out_csv = out_dir / "candidates_step5_filtered.csv"
candidates_step5_filtered.to_csv(out_csv, index=False)
print("Saved:", out_csv)

display(candidates_step5_filtered.head(20))


Sample result keys: ['tokens', 'activeFeatures']
Token-ish keys: ['tokens']
Prompts missing tokens: 0 / 30


,feature_id,score_boost,score_suppress,score_answerable,count_myth_tau,count_fact_tau,count_unanswerable_tau,count_total_tau,freq_myth_tau,freq_fact_tau,...,mass_total,mass_myth,mass_fact,mass_unanswerable,max_mass_myth,max_mass_fact,max_mass_unanswerable,punct_mass_ratio,early_mass_ratio,prompts_counted_step5
7,100246,0.8,0.2,0.30,10,0,2,12,1.0,0.0,...,5638.0,4846.0,0.0,792.0,696.0,0.0,396.0,1.000000,0.198652,12
37,173576,0.3,0.7,-0.20,10,0,7,17,1.0,0.0,...,13144.0,9032.0,0.0,4112.0,1096.0,0.0,824.0,1.000000,0.141814,17
23,223643,0.7,0.0,0.35,7,0,0,7,0.7,0.0,...,2438.0,2438.0,0.0,0.0,512.0,0.0,0.0,1.000000,0.000000,7
53,6878,-0.1,-0.5,0.45,4,5,0,9,0.4,0.5,...,2272.0,936.0,1336.0,0.0,288.0,416.0,0.0,1.000000,0.077465,9
54,70353,-0.2,-0.5,0.40,4,6,1,11,0.4,0.6,...,4308.0,1528.0,2380.0,400.0,544.0,512.0,400.0,1.000000,0.000000,11
69,180797,-0.7,0.7,-0.70,2,2,9,13,0.2,0.2,...,5504.0,792.0,1192.0,3520.0,440.0,864.0,472.0,1.000000,0.094477,13
63,4448,-0.5,0.6,-0.55,1,0,6,7,0.1,0.0,...,4064.0,208.0,0.0,3856.0,208.0,0.0,1680.0,1.000000,0.000000,7
74,11974,-0.9,0.9,-0.90,0,0,9,9,0.0,0.0,...,3524.0,0.0,0.0,3524.0,0.0,0.0,692.0,1.000000,0.112372,9
35,7668,0.4,-0.3,0.50,8,4,1,13,0.8,0.4,...,8036.0,4928.0,2812.0,296.0,1264.0,1276.0,296.0,1.000000,0.232952,13
58,10718,-0.4,-0.6,0.40,6,10,4,20,0.6,1.0,...,6089.0,1250.0,3919.0,920.0,324.0,710.0,462.0,0.978157,0.195434,20


Step 5 filtered: 75 -> 49
Dropped due to punct > 0.6 : 26
Saved: /content/drive/MyDrive/cs175/neuronpedia_runs/candidates_step5_filtered.csv


,feature_id,score_boost,score_suppress,score_answerable,count_myth_tau,count_fact_tau,count_unanswerable_tau,count_total_tau,freq_myth_tau,freq_fact_tau,...,mass_total,mass_myth,mass_fact,mass_unanswerable,max_mass_myth,max_mass_fact,max_mass_unanswerable,punct_mass_ratio,early_mass_ratio,prompts_counted_step5
0,31003,1.0,0.0,0.50,10,0,0,10,1.0,0.0,...,5604.0,5604.0,0.0,0.0,780.0,0.0,0.0,0.000000,0.130621,10
1,6258,1.0,0.0,0.50,10,0,0,10,1.0,0.0,...,3884.0,3884.0,0.0,0.0,454.0,0.0,0.0,0.000000,0.096292,10
2,34723,0.8,0.2,0.30,10,0,2,12,1.0,0.0,...,14156.0,13520.0,0.0,636.0,1920.0,0.0,368.0,0.573326,0.106810,12
3,20479,0.8,0.1,0.35,9,0,1,10,0.9,0.0,...,4322.0,3972.0,0.0,350.0,680.0,0.0,350.0,0.000000,0.144378,10
4,71403,0.8,-0.2,0.60,10,2,0,12,1.0,0.2,...,20848.0,20252.0,596.0,0.0,2792.0,316.0,0.0,0.380660,0.059862,12
5,32469,0.7,0.1,0.30,8,0,1,9,0.8,0.0,...,2474.0,2208.0,0.0,266.0,346.0,0.0,266.0,0.000000,0.112369,9
6,3757,0.7,0.0,0.35,8,1,1,10,0.8,0.1,...,3464.0,2720.0,310.0,434.0,402.0,310.0,434.0,0.000000,0.301963,10
7,9374,0.7,0.0,0.35,8,1,1,10,0.8,0.1,...,3313.0,2719.0,378.0,216.0,522.0,378.0,216.0,0.000000,0.271053,10
8,18061,0.7,-0.1,0.45,8,1,0,9,0.8,0.1,...,2794.0,2512.0,282.0,0.0,346.0,282.0,0.0,0.100931,0.100931,9
9,115974,0.7,0.2,0.25,9,0,2,11,0.9,0.0,...,4504.0,3912.0,0.0,592.0,720.0,0.0,312.0,0.000000,0.563055,11


In [ ]:
# ============================================
# Step 6 — Pull feature metadata for interpretability + manual tags
# Input: candidates_step5_filtered (preferred) OR candidates_step4
# Output: candidates_with_metadata.csv + candidates_with_metadata.jsonl
# ============================================

import time
import json

BASE = "https://www.neuronpedia.org"
MODEL_ID = MODEL_ID
SOURCE = SOURCE


FINAL_K = 3


final_boost = top_boost.head(FINAL_K).copy()
final_boost["candidate_type"] = "boost"

final_suppress = top_suppress.head(FINAL_K).copy()
final_suppress["candidate_type"] = "suppress"

final_answerable = top_answerable.head(FINAL_K).copy()
final_answerable["candidate_type"] = "answerable"

final_balanced = top_balanced_all3.head(FINAL_K).copy()
final_balanced["candidate_type"] = "balanced"


base_candidates = pd.concat(
    [final_boost, final_suppress, final_answerable, final_balanced],
    axis=0
).reset_index(drop=True)

print("Using category-aware final candidates from Step 4")
print(base_candidates["candidate_type"].value_counts(dropna=False))

MAX_FEATURES_TO_FETCH = len(base_candidates)


def get_np(path, retries=5, sleep=0.8, timeout=60):
    url = f"{BASE}{path}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Colab)",
    }

    if "NEURONPEDIA_TOKEN" in globals() and NEURONPEDIA_TOKEN:
        headers["X-API-Key"] = NEURONPEDIA_TOKEN

    backoff = sleep
    last_err = None

    for _ in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            if r.status_code == 200:
                return r.json()
            if r.status_code in (429, 500, 502, 503, 504):
                last_err = f"HTTP {r.status_code}: {r.text[:300]}"
                time.sleep(backoff)
                backoff *= 2
                continue
            raise RuntimeError(f"HTTP {r.status_code} {path}: {r.text[:800]}")
        except requests.RequestException as e:
            last_err = f"RequestException: {str(e)[:300]}"
            time.sleep(backoff)
            backoff *= 2
            continue

    raise RuntimeError(f"Failed GET after retries. Last error: {last_err}")

def fetch_feature_json(model_id, source, feature_id):

    return get_np(f"/api/feature/{model_id}/{source}/{int(feature_id)}")


def pick_first_str(d, keys):
    for k in keys:
        v = d.get(k, None)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return ""

def summarize_examples(obj, max_examples=3, max_chars=240):
    examples = []


    candidate_paths = [
        ("topActivations",),
        ("top_activations",),
        ("activations", "top"),
        ("examples",),
        ("feature", "topActivations"),
    ]

    def try_get_path(o, path):
        cur = o
        for p in path:
            if not isinstance(cur, dict) or p not in cur:
                return None
            cur = cur[p]
        return cur

    def normalize_item(item):

        if isinstance(item, str):
            return item
        if isinstance(item, dict):
            for k in ["text", "prompt", "content", "example", "decoded", "input"]:
                if k in item and isinstance(item[k], str) and item[k].strip():
                    return item[k].strip()
        return None

    for path in candidate_paths:
        got = try_get_path(obj, path)
        if isinstance(got, list):
            for it in got:
                s = normalize_item(it)
                if s:
                    examples.append(s)
        if examples:
            break

    if not examples:
        def walk(o, depth=0, max_depth=4):
            if depth > max_depth:
                return
            if isinstance(o, dict):
                for _, v in o.items():
                    walk(v, depth+1, max_depth)
            elif isinstance(o, list):

                for v in o[:50]:
                    if isinstance(v, dict) and any(k in v for k in ["text","prompt","content"]):
                        s = normalize_item(v)
                        if s:
                            examples.append(s)
                    walk(v, depth+1, max_depth)

        walk(obj)

    examples = [e.replace("\n", " ").strip() for e in examples if e.strip()]
    examples = list(dict.fromkeys(examples))

    short = []
    for e in examples[:max_examples]:
        short.append(e[:max_chars] + ("…" if len(e) > max_chars else ""))

    return " | ".join(short), len(examples)

def extract_autointerp(obj):

    if isinstance(obj.get("autointerp"), dict):
        ai = obj["autointerp"]
        return (
            pick_first_str(ai, ["text", "explanation", "description"]),
            ai.get("score", None),
            pick_first_str(ai, ["model", "modelName", "autoInterpType"]),
        )


    exps = obj.get("explanations", None)
    if isinstance(exps, list) and exps:

        def score_of(e):
            if isinstance(e, dict):
                s = e.get("score", None)
                return -1 if s is None else s
            return -1
        best = max([e for e in exps if isinstance(e, dict)], key=score_of, default=None)
        if best:
            txt = pick_first_str(best, ["text", "explanation", "description"])
            score = best.get("score", None)
            model = pick_first_str(best, ["model", "modelName", "autoInterpType"])
            return txt, score, model


    return "", None, ""


rows = []
raw_jsonl = []

feature_ids_to_fetch = base_candidates["feature_id"].drop_duplicates().tolist()
for fid in tqdm(feature_ids_to_fetch, desc="Fetching /api/feature"):
    try:
        js = fetch_feature_json(MODEL_ID, SOURCE, fid)
    except Exception as e:
        rows.append({
            "feature_id": int(fid),
            "meta_ok": False,
            "meta_error": str(e)[:300],
            "autointerp_text": "",
            "autointerp_score": None,
            "autointerp_model": "",
            "top_examples_preview": "",
            "n_examples_found": 0,
        })
        continue

    aut_text, aut_score, aut_model = extract_autointerp(js)
    ex_preview, n_ex = summarize_examples(js, max_examples=3)

    rows.append({
        "feature_id": int(fid),
        "meta_ok": True,
        "meta_error": "",
        "autointerp_text": aut_text,
        "autointerp_score": aut_score,
        "autointerp_model": aut_model,
        "top_examples_preview": ex_preview,
        "n_examples_found": int(n_ex),
    })

    raw_jsonl.append({"feature_id": int(fid), "json": js})


df_meta = pd.DataFrame(rows)
candidates_with_meta = base_candidates.merge(df_meta, on="feature_id", how="left")


for col in ["looks_refusal", "looks_topic_specific", "looks_formatting", "looks_correction", "notes"]:
    if col not in candidates_with_meta.columns:
        candidates_with_meta[col] = ""

display(candidates_with_meta.head(20))

boost_with_meta = candidates_with_meta[candidates_with_meta["candidate_type"] == "boost"].copy()
suppress_with_meta = candidates_with_meta[candidates_with_meta["candidate_type"] == "suppress"].copy()
answerable_with_meta = candidates_with_meta[candidates_with_meta["candidate_type"] == "answerable"].copy()
balanced_with_meta = candidates_with_meta[candidates_with_meta["candidate_type"] == "balanced"].copy()

boost_csv_path = out_dir / "boost_candidates_with_metadata.csv"
suppress_csv_path = out_dir / "suppress_candidates_with_metadata.csv"
answerable_csv_path = out_dir / "answerable_candidates_with_metadata.csv"
balanced_csv_path = out_dir / "balanced_candidates_with_metadata.csv"

boost_with_meta.to_csv(boost_csv_path, index=False)
suppress_with_meta.to_csv(suppress_csv_path, index=False)
answerable_with_meta.to_csv(answerable_csv_path, index=False)
balanced_with_meta.to_csv(balanced_csv_path, index=False)

out_dir = Path("/content/drive/MyDrive/cs175/neuronpedia_runs")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "candidates_with_metadata.csv"
jsonl_path = out_dir / "candidates_with_metadata.raw.jsonl"

candidates_with_meta.to_csv(csv_path, index=False)

with jsonl_path.open("w", encoding="utf-8") as f:
    for r in raw_jsonl:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("Saved:")
print(" ", csv_path)
print(" ", jsonl_path)


Using category-aware final candidates from Step 4
candidate_type
boost         3
suppress      3
answerable    3
balanced      3
Name: count, dtype: int64


Fetching /api/feature: 100%|██████████| 12/12 [00:14<00:00,  1.18s/it]


,feature_id,count_myth_tau,mass_myth,max_mass_myth,freq_myth_tau,count_fact_tau,mass_fact,max_mass_fact,freq_fact_tau,count_unanswerable_tau,...,autointerp_text,autointerp_score,autointerp_model,top_examples_preview,n_examples_found,looks_refusal,looks_topic_specific,looks_formatting,looks_correction,notes
0,31003,10,5604.0,780.0,1.0,0,0.0,0.0,0.0,0,...,ingredients with quantities,None,,,0,,,,,
1,6258,10,3884.0,454.0,1.0,0,0.0,0.0,0.0,0,...,complex or challenging,None,,,0,,,,,
2,11591,9,12596.0,2176.0,0.9,0,0.0,0.0,0.0,0,...,studies and research,None,,,0,,,,,
3,11143,4,2278.0,762.0,0.4,1,864.0,864.0,0.1,10,...,first person pronoun,None,,,0,,,,,
4,11974,0,0.0,0.0,0.0,0,0.0,0.0,0.0,9,...,incredible passing,None,,,0,,,,,
5,4351,9,3516.0,648.0,0.9,1,500.0,500.0,0.1,9,...,hello and punctuation,None,,,0,,,,,
6,8720,10,5636.0,1236.0,1.0,5,11112.0,6080.0,0.5,0,...,biological life and processes,None,,,0,,,,,
7,29586,10,3736.0,442.0,1.0,7,2162.0,406.0,0.7,1,...,offering examples or explanations,None,,,0,,,,,
8,5265,8,3266.0,958.0,0.8,8,7647.0,1381.0,0.8,2,...,grammar and sentence structure,None,,,0,,,,,
9,8366,10,65362.0,9300.0,1.0,10,75850.0,9762.0,1.0,10,...,write requests,None,,,0,,,,,


Saved:
  /content/drive/MyDrive/cs175/neuronpedia_runs/candidates_with_metadata.csv
  /content/drive/MyDrive/cs175/neuronpedia_runs/candidates_with_metadata.raw.jsonl
